## 1. Futures prices — raw structure

The file is not a flat table. It is a raw API extract: an OData envelope with
three top-level fields — `@odata.context`, `Contents` (the records), and
`Notes` (the provider's extraction log). The 408 records sit inside the
`Contents` array, so the first step of any transformation is
`unnest(Contents, recursive := true)`.

In [1]:
import duckdb
con = duckdb.connect()
raw = "../data/bronze/extracted/data_assignment"
con.sql(f"""
select * from (
    select unnest(Contents, recursive := true)
    from read_json_auto('{raw}/futures_prices.json')
)
""")

┌───────────┬────────────┬───────────────────────┬────────────┬─────────────────┬───────────┬───────────────┬────────┬─────────────────────────────┬───────────────┬──────────────────────────┬───────────────┬─────────────────────┬─────────────────────┬───────────────┬─────────────────────────┬──────────────┬───────────┐
│ Low Price │ High Price │ Universal Close Price │ Trade Date │ Expiration Date │ Lot Units │ Currency Code │ Volume │ Accumulated Volume Unscaled │ Exchange Code │   Exchange Description   │ Open Interest │ Universal Ask Price │ Universal Bid Price │ Market Volume │ Contract Month and Year │ Product Code │   Error   │
│   int64   │   int64    │        double         │    date    │      date       │  varchar  │    varchar    │  json  │            int64            │    varchar    │         varchar          │     int64     │        int64        │        int64        │     int64     │         varchar         │   varchar    │  varchar  │
├───────────┼────────────┼───────────

**Observations**

1. 408 entries. Four of them carry only `Error = 'Not found'` — failed request
   items in the extract, not data.
2. Four further rows have no `Trade Date` and no close price. All four are
   FEB2026 contracts that expired before the sample week (expiration
   2026-02-20/25). Same symptom as the error stubs, different cause.
3. `Lot Units` is mixed: TONNE for FASM/FABT/FAWH, KG100 for FALM. Prices are
   quoted per lot unit, so KG100 prices need a ×10 step to be per metric tonne.
4. `Low Price`, `High Price` and `Volume` are almost entirely null, and none of
   them are needed downstream.

**Decisions for `stg_futures_prices`**

- Drop the 4 error stubs and the 4 expired-contract rows. Both exclusions are
  handled separately, because they have different causes.
- Staging renames and types only. The KG100→MT conversion happens in the mart,
  next to the currency conversion, so all price arithmetic lives in one place.
- Select only the columns the final table needs.

In [2]:
# local reading convenience only: a named SELECT over the raw file
con.sql(f"""
create or replace view raw_futures as
select unnest(Contents, recursive := true)
from read_json_auto('{raw}/futures_prices.json')
""")

1. 408 entries. Four of them carry only `Error = 'Not found'` — failed request
   items in the extract, not data.

In [3]:
con.sql('select "Error", count(*) as n from raw_futures group by 1')

┌───────────┬───────┐
│   Error   │   n   │
│  varchar  │ int64 │
├───────────┼───────┤
│ NULL      │   404 │
│ Not found │     4 │
└───────────┴───────┘

In [4]:
con.sql('''
select "Error", "Trade Date", "Product Code", "Universal Close Price"
from raw_futures
where "Error" is not null
''')

┌───────────┬────────────┬──────────────┬───────────────────────┐
│   Error   │ Trade Date │ Product Code │ Universal Close Price │
│  varchar  │    date    │   varchar    │        double         │
├───────────┼────────────┼──────────────┼───────────────────────┤
│ Not found │ NULL       │ NULL         │                  NULL │
│ Not found │ NULL       │ NULL         │                  NULL │
│ Not found │ NULL       │ NULL         │                  NULL │
│ Not found │ NULL       │ NULL         │                  NULL │
└───────────┴────────────┴──────────────┴───────────────────────┘

2. Four further rows have no `Trade Date` and no close price. All four are
   FEB2026 contracts that expired before the sample week (expiration
   2026-02-20/25). Same symptom as the error stubs, different cause.

In [5]:
con.sql('''
select "Product Code", "Contract Month and Year", "Expiration Date",
       "Trade Date", "Universal Close Price"
from raw_futures
where "Error" is null and "Trade Date" is null
''')

┌──────────────┬─────────────────────────┬─────────────────┬────────────┬───────────────────────┐
│ Product Code │ Contract Month and Year │ Expiration Date │ Trade Date │ Universal Close Price │
│   varchar    │         varchar         │      date       │    date    │        double         │
├──────────────┼─────────────────────────┼─────────────────┼────────────┼───────────────────────┤
│ FASM         │ FEB2026                 │ 2026-02-25      │ NULL       │                  NULL │
│ FABT         │ FEB2026                 │ 2026-02-25      │ NULL       │                  NULL │
│ FAWH         │ FEB2026                 │ 2026-02-25      │ NULL       │                  NULL │
│ FALM         │ FEB2026                 │ 2026-02-20      │ NULL       │                  NULL │
└──────────────┴─────────────────────────┴─────────────────┴────────────┴───────────────────────┘

3. `Lot Units` is mixed: TONNE for FASM/FABT/FAWH, KG100 for FALM. Prices are
   quoted per lot unit, so KG100 prices need a ×10 step to be per metric tonne.

In [6]:
con.sql('''
select "Product Code", "Lot Units", count(*) as n,
       round(avg("Universal Close Price"), 2) as avg_close
from raw_futures
where "Error" is null and "Trade Date" is not null
group by 1, 2
order by 1
''')

┌──────────────┬───────────┬───────┬───────────┐
│ Product Code │ Lot Units │   n   │ avg_close │
│   varchar    │  varchar  │ int64 │  double   │
├──────────────┼───────────┼───────┼───────────┤
│ FABT         │ TONNE     │   100 │   5262.27 │
│ FALM         │ KG100     │   100 │     44.48 │
│ FASM         │ TONNE     │   100 │   2738.29 │
│ FAWH         │ TONNE     │   100 │   1381.03 │
└──────────────┴───────────┴───────┴───────────┘

4. `Low Price`, `High Price` and `Volume` are almost entirely null, and none of
   them are needed downstream.

In [7]:
con.sql('''
select count(*) as total_rows,
       count("Low Price")     as low_price_filled,
       count("High Price")    as high_price_filled,
       count("Volume")        as volume_filled,
       count("Open Interest") as open_interest_filled
from raw_futures
where "Error" is null
''')

┌────────────┬──────────────────┬───────────────────┬───────────────┬──────────────────────┐
│ total_rows │ low_price_filled │ high_price_filled │ volume_filled │ open_interest_filled │
│   int64    │      int64       │       int64       │     int64     │        int64         │
├────────────┼──────────────────┼───────────────────┼───────────────┼──────────────────────┤
│        404 │                3 │                 3 │             0 │                  152 │
└────────────┴──────────────────┴───────────────────┴───────────────┴──────────────────────┘

## 2. Curated layer — `fct_futures_prices`

From this point on, the notebook reads from the dbt-built DuckDB database instead
of the raw files. All transformations (error-row filtering, unit normalisation,
FX conversion, product mapping) live in dbt models under `transform/` — the
notebook only observes their output.

The `q()` helper below opens a **read-only, per-query** connection: DuckDB allows
a single writer per database file, so keeping the notebook read-only means
`dbt run` can rebuild models at any time without lock conflicts.

In [8]:
def q(sql):
    with duckdb.connect("../transform/interfood_dev.duckdb", read_only=True) as c:
        return c.sql(sql).df()

### Acceptance checks

One query, seven assertions against the final mart:

1. `row_count = 400` — left joins must preserve the row count of the cleaned
   futures set; any drift would mean a duplicate join key.
2. `null_usd = 0` and `null_product = 0` — every contract month found an FX rate,
   every product code found a mapping entry.
3. `sub100_rows = 0` and `min_price_local ≈ 386.3` — the 100KG-quoted product
   (previously 38.63 per 100KG) has been rescaled ×10 to per-MT; no sub-100
   stragglers remain.
4. `ratio_min/max ∈ [1.16, 1.20]` — the implied EUR→USD rate per row sits inside
   the forward-curve range, confirming the conversion was a multiplication and
   not an inversion.

In [9]:
q('''
select
    count(*)                                              as row_count,
    count(*) filter (where "PriceUSDMT" is null)          as null_usd,
    count(*) filter (where "Product" is null)             as null_product,
    count(*) filter (where "PriceLocal" < 100)            as sub100_rows,
    round(min("PriceLocal"), 2)                           as min_price_local,
    round(min("PriceUSDMT" / "PriceLocal"), 3)            as ratio_min,
    round(max("PriceUSDMT" / "PriceLocal"), 3)            as ratio_max
from fct_futures_prices
''')

,row_count,null_usd,null_product,sub100_rows,min_price_local,ratio_min,ratio_max
0,400,0,0,0,386.3,1.164,1.195


In [10]:
q('''
select *
from fct_futures_prices
''')

,Timestamp,Market,Product,UnitOfMeasure,Currency,PriceLocal,PriceUSDMT,Period,Month,Year,IngestionTimestamp
0,2026-03-09,EEX,Milk,MT,EUR,464.1,551.982429,2027-06-21,6,2027,2026-08-24 16:50:20.491893
1,2026-03-09,EEX,Milk,MT,EUR,464.1,552.635228,2027-07-20,7,2027,2026-08-24 16:50:20.491893
2,2026-03-09,EEX,Milk,MT,EUR,464.1,553.288026,2027-08-20,8,2027,2026-08-24 16:50:20.491893
3,2026-03-09,EEX,Milk,MT,EUR,464.1,553.940825,2027-09-20,9,2027,2026-08-24 16:50:20.491893
4,2026-03-09,EEX,Milk,MT,EUR,464.1,554.593623,2027-10-20,10,2027,2026-08-24 16:50:20.491893
...,...,...,...,...,...,...,...,...,...,...,...
395,2026-03-13,EEX,SMP,MT,EUR,2800.0,3321.660471,2027-04-28,4,2027,2026-08-24 16:50:20.491893
396,2026-03-13,EEX,SMP,MT,EUR,2800.0,3325.935602,2027-05-26,5,2027,2026-08-24 16:50:20.491893
397,2026-03-13,EEX,SMP,MT,EUR,2607.0,3035.766652,2026-03-25,3,2026,2026-08-24 16:50:20.491893
398,2026-03-13,EEX,SMP,MT,EUR,2781.0,3244.011338,2026-04-29,4,2026,2026-08-24 16:50:20.491893


### Market & product coverage

Before computing the required metric, one look at what the mart actually
contains: a single market (EEX) and four products. The mapping table covers
26 product codes, but this week's feed only uses four of them — the
`relationships` test in `schema.yml` guards exactly this direction
(every code in the feed must exist in the mapping; unused mapping entries
are the normal surplus of a reference table, not an issue).

Practical consequence: the metric filter below still states
`Market = 'EEX'` explicitly. It is redundant today, but it declares the
intended scope and stays correct the day a second market (e.g. NZX) lands
in this pipeline.

In [11]:
q('select distinct "Market", "Product" from fct_futures_prices order by 1, 2')

,Market,Product
0,EEX,Butter
1,EEX,Milk
2,EEX,SMP
3,EEX,SWP Feed


## 3. Required output 1 — average price change, March 2026 SMP (EEX)

The phrase *"average price change for March 2026"* allows two readings:

- **Reading A — contract month.** "March 2026" names the MAR2026 contract:
  the average day-over-day change of that single contract across the trade
  days in the sample.
- **Reading B — trading period.** "March 2026" names the period: the average
  day-over-day change of *all* SMP EEX contracts quoted during these days.

**Reading A is the primary answer.** The source data spells contracts exactly
as `MAR2026`, the assignment echoes that format, and in market language
"the March contract" refers to the contract expiring in March. Reading B is
reported below it as a sensitivity check — it blends near and far tenors,
which barely move, and so dilutes the figure.

Definition of "average change": the mean of daily differences in
`PriceUSDMT`, ordered by trade date. The first day has no predecessor and
is naturally excluded.

In [12]:
q('''
with mar_contract as (
    select "Timestamp", "PriceUSDMT"
    from fct_futures_prices
    where "Market"  = 'EEX' 
      and "Product" = 'SMP'
      and "Year"    = 2026
      and "Month"   = 3
),
daily as (
    select
        "Timestamp",
        "PriceUSDMT",
        "PriceUSDMT" - lag("PriceUSDMT") over (order by "Timestamp") as day_change
    from mar_contract
)
select * from daily order by "Timestamp"
''')

,Timestamp,PriceUSDMT,day_change
0,2026-03-09,3042.753457,NaN
1,2026-03-10,3031.108783,-11.644675
2,2026-03-11,3043.917925,12.809142
3,2026-03-12,3054.398132,10.480207
4,2026-03-13,3035.766652,-18.631479


### Reading A — result

`avg()` ignores the null on the first day; `n_daily_changes = 4` confirms
exactly four differences enter the mean.

Cross-check: a mean of daily differences telescopes to
*(last − first) / n*. Here: (3035.77 − 3042.75) / 4 ≈ **−1.75 USD/MT per
trading day**, matching the query — the week's cumulative move is −6.99
USD/MT. Two routes, one number.

In [13]:
q('''
with mar_contract as (
    select "Timestamp", "PriceUSDMT"
    from fct_futures_prices
    where "Market"  = 'EEX'
      and "Product" = 'SMP'
      and "Year"    = 2026
      and "Month"   = 3
),
daily as (
    select
        "PriceUSDMT" - lag("PriceUSDMT") over (order by "Timestamp") as day_change
    from mar_contract
)
select
    round(avg(day_change), 2)  as avg_daily_change_usd_per_mt,
    count(day_change)          as n_daily_changes
from daily
''')

,avg_daily_change_usd_per_mt,n_daily_changes
0,-1.75,4


### Reading B — sensitivity check

Same skeleton, two changes: the contract filter is dropped, and the window
gains `partition by "Year", "Month"` so each contract computes differences
against **its own** previous day — without the partition, `lag` would
subtract prices across different contracts, which is meaningless.

Expectation before running: 20 SMP contracts × 4 differences = 80 changes,
and a mean closer to zero than Reading A, because far tenors barely move
during the week. That dilution is precisely why Reading A is the primary
answer.

**Result — expectation falsified.** Reading B came out at **+23.95 USD/MT
per day**, larger than Reading A and opposite in sign. A per-contract
breakdown (below) shows why: MAR2026 is the *only* contract that fell this
week; every other tenor rallied by roughly +100 USD/MT cumulatively. The
"far tenors barely move" assumption was wrong — the whole curve shifted up
while the expiring front month drifted down. The near-identical weekly moves
across all 2027 tenors (+106.4 … +107.6) suggest far-month quotes are
carried/interpolated rather than independently traded.

This strengthens the case for Reading A on different grounds: A and B answer
different questions. A describes the named contract; B averages a broad
curve repricing that the question, as worded, did not ask about.

In [14]:
q('''
with smp as (
    select "Timestamp", "Year", "Month", "PriceUSDMT"
    from fct_futures_prices
    where "Market"  = 'EEX'
      and "Product" = 'SMP'
),
daily as (
    select
        "PriceUSDMT" - lag("PriceUSDMT") over (
            partition by "Year", "Month"
            order by "Timestamp"
        ) as day_change
    from smp
)
select
    round(avg(day_change), 2)  as avg_daily_change_usd_per_mt,
    count(day_change)          as n_daily_changes
from daily
''')

,avg_daily_change_usd_per_mt,n_daily_changes
0,23.95,80


In [15]:
q('''
with smp as (
    select "Timestamp", "Year", "Month", "PriceUSDMT"
    from fct_futures_prices
    where "Market" = 'EEX' and "Product" = 'SMP'
),
daily as (
    select "Year", "Month",
        "PriceUSDMT" - lag("PriceUSDMT") over (
            partition by "Year", "Month" order by "Timestamp"
        ) as day_change
    from smp
)
select "Year", "Month",
    round(avg(day_change), 2) as avg_daily_change,
    round(sum(day_change), 2) as week_change
from daily
group by 1, 2
order by 1, 2
''')

,Year,Month,avg_daily_change,week_change
0,2026,3,-1.75,-6.99
1,2026,4,23.62,94.49
2,2026,5,29.21,116.85
3,2026,6,31.90,127.59
4,2026,7,29.30,117.21
5,2026,8,29.34,117.38
6,2026,9,37.32,149.27
7,2026,10,7.36,29.42
8,2026,11,13.26,53.04
9,2026,12,12.10,48.39


## 4. Required output 2 — forward-filling the next trade day

The sample ends on Friday 2026-03-13. The assignment asks for rows for the
next trade day, 2026-03-16, in the same CSV. Standard market-data practice:
when no new quote exists, carry the last known one forward.

This is a pipeline action, so it lives in dbt, not in this notebook:
`fct_futures_prices_filled` = the 400 observed rows, plus a copy of the
final day's 80 rows re-stamped with the next trade day. The date is
**derived** (last date + weekend skip), not hard-coded — the model stays
correct when new data arrives. Holidays are out of scope here; a production
version would join an exchange calendar (see README, improvements).

`fct_futures_prices` itself stays untouched: one table holds only observed
prices, the other adds the synthetic fill — two tables, two clear meanings.

### Why the fill stops at one day

The dataset's "now" is its last observation, Friday 2026-03-13 — not the
date this notebook happens to run. The model therefore anchors on
`max(Timestamp)` rather than `current_date`: it always bridges exactly one
step past the data, never from the sample to the wall clock. Re-running the
pipeline months later still yields 2026-03-16 — a re-run must not fabricate
months of prices.

Forward-filled values are placeholders, not data: their credibility decays
with distance. Bridging a weekend is standard practice; bridging five months
would assert an unmoving market. When real 03-16 settlements arrive through
ingestion, the anchor moves forward and the bridge moves with it.

Acceptance below: 480 rows total, 80 filled, max date 2026-03-16.

In [16]:
q('''
select
    count(*)                                             as total_rows,
    count(*) filter (where "Timestamp" = date '2026-03-16') as filled_rows,
    max("Timestamp")                                     as max_ts
from fct_futures_prices_filled
''')

,total_rows,filled_rows,max_ts
0,480,80,2026-03-16
